## Peform a search

> The OneAtlas Data search API is compliant with the OpenSearch specification. The official specification documentation can be used as a reference: http://www.opensearch.org/.

In [1]:
import json, requests

import pandas as pd
import geopandas as gpd

from airbus_helper import *
from tqdm.notebook import tqdm

%load_ext autoreload
%autoreload 2

In [2]:
SEARCH_API_ENDPOINT = 'https://search.foundation.api.oneatlas.airbus.com/api/v2/opensearch'

api_token = get_api_key("AIRBUS") #input("Enter your access token")
access_token = get_airbus_token(api_token)
print(f"Access token obtained successfully\n")

Access token obtained successfully



In [22]:
# load input data
input_folder = r"C:\Users\WB411133\OneDrive - WBG\AAA_BPS\GOST\Projects\IMAGERY_RFP\Purchases\2026-08-31_GHA_roads"
unq_id_col = None # Defines a primary key in the source data
output_folder = os.path.join(input_folder, "AIRBUS_SEARCH")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
roads_file = os.path.join(input_folder, "validated_segments_C1.geojson")
out_roads_file = os.path.join(output_folder, "validated_segments_C1_buffered.geojson")

in_roads = gpd.read_file(roads_file)
if not unq_id_col:
    # if no primary key exists, create one using the index
    unq_id_col = 'uid'
    in_roads[unq_id_col] = in_roads.index

# roads are lines, so they need to be buffered to create polygons for spatial operations
in_roads = in_roads.to_crs(3857)
in_roads['geometry'] = in_roads.buffer(1000)  
in_roads = in_roads.to_crs(4326)
if not os.path.exists(out_roads_file):
    in_roads.to_file(out_roads_file, driver='GeoJSON')

search_gdf = in_roads.copy()

In [24]:
all_res = {}
clip_shp_extent = True # set to True to clip the searched imagery extents to the source shape

# For each port, search for images within a 2km radius
all_image_gdf = []
for idx, row in tqdm(search_gdf.iterrows(), total=search_gdf.shape[0]):    
    cur_bbox = ",".join(map(str, row.geometry.bounds))
    
    # Perform the search
    image_gdf = search_airbus_imagery(access_token, cur_bbox)
    
    # Calculate %overlap of each image with the port buffer
    image_gdf['overlap_pct'] = image_gdf.geometry.apply(lambda x: x.intersection(row.geometry).area / row.geometry.area * 100)
    
    # Drop images with less than 50% overlap
    image_gdf = image_gdf[image_gdf['overlap_pct'] >= 50]

    # Calculate year of acquisition for each image
    image_gdf['acquisition_year'] = image_gdf['acquisitionDate'].dt.year

    if clip_shp_extent:
        image_gdf['geometry'] = image_gdf.geometry.intersection(row.geometry)

    image_gdf.to_file(f"{output_folder}/{row[unq_id_col]}_airbus_imagery.geojson", driver='GeoJSON')
    all_image_gdf.append(image_gdf)

    # Calculate the number of images acquired each year
    images_per_year = image_gdf.groupby('acquisition_year').size()
    all_res[row[unq_id_col]] = images_per_year


# Write out summary files
pd.DataFrame(all_res).T.to_csv(f"{output_folder}/airbus_imagery_summary.csv")
if all_image_gdf:
    all_image_gdf = gpd.GeoDataFrame(pd.concat(all_image_gdf, ignore_index=True))
    all_image_gdf.to_file(f"{output_folder}/all_airbus_imagery.geojson", driver='GeoJSON')

    

  0%|          | 0/27 [00:00<?, ?it/s]